# ml_D_svm_codes.ipynb

**所属章节**：Chapter D 支持向量机  
**用途**：生成讲义中的全部配图（5张），保存至 `figs/` 目录  
**运行说明**：顺序执行，约 2–3 分钟

**输出文件**：
- `figs/fig_D_svm_margin.png`：最大间隔分类器示意
- `figs/fig_D_soft_margin.png`：不同 C 值下的软间隔决策边界
- `figs/fig_D_kernel_trick.png`：核方法示意（2D → 3D）
- `figs/fig_D_rbf_gamma.png`：RBF 核不同 gamma 值下的决策边界
- `figs/fig_D_svm_vs_lr.png`：SVM vs Logistic 回归决策边界对比


In [ ]:
# Global setup
import os
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.mplconfig'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification, make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

# Chinese font support
from matplotlib import font_manager

candidate_fonts = [
    'Microsoft YaHei', 'SimHei', 'SimSun', 'Arial Unicode MS',
    'Noto Sans CJK SC', 'Source Han Sans SC', 'WenQuanYi Micro Hei'
]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
zh_fonts = [f for f in candidate_fonts if f in available_fonts]
ZH_FONT = zh_fonts[0] if zh_fonts else None
if zh_fonts:
    plt.rcParams['font.sans-serif'] = zh_fonts + plt.rcParams['font.sans-serif']
else:
    print('Warning: no common Chinese font found; Chinese text may render as boxes.')
FP = font_manager.FontProperties(family=ZH_FONT) if ZH_FONT else font_manager.FontProperties()
FPB = font_manager.FontProperties(family=ZH_FONT, weight='bold') if ZH_FONT else font_manager.FontProperties(weight='bold')
plt.rcParams['axes.unicode_minus'] = False

C_color = {
    'primary':'#0B3D91','secondary':'#B8860B','tertiary':'#2F5E9E',
    'neutral':'#878787','highlight':'#6B4E00','fill':'#D6E2F3',
    'cls0':'#0B3D91','cls1':'#B8860B',
}
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
    'font.size':11,'axes.titlesize':13,'axes.labelsize':11,
    'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)

# Helper function for decision boundary plots
def plot_decision_boundary(ax, model, X, y, title='', show_sv=True):
    x_min,x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min,y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min,x_max,300),
                          np.linspace(y_min,y_max,300))
    Z = model.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z, alpha=0.15,
                colors=[C_color['cls0'],C_color['cls1']])
    ax.contour(xx,yy,Z, colors='k', linewidths=1.2)
    ax.scatter(X[y==0,0],X[y==0,1], color=C_color['cls0'],
               edgecolors='white', s=30, lw=0.5, zorder=3)
    ax.scatter(X[y==1,0],X[y==1,1], color=C_color['cls1'],
               edgecolors='white', s=30, marker='^', lw=0.5, zorder=3)
    if show_sv and hasattr(model,'support_vectors_'):
        sv = model.support_vectors_
        ax.scatter(sv[:,0],sv[:,1], s=120, facecolors='none',
                   edgecolors='k', linewidths=1.5, zorder=4,
                   label=f'Support vectors ({len(sv)})')
    ax.set_title(title, fontproperties=FPB, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

print('Global setup complete')


---
## 图1：最大间隔分类器

线性可分数据，用硬间隔 SVM（C 极大）展示最大间隔、决策边界和支持向量。


In [ ]:
# 生成线性可分数据
np.random.seed(SEED)
X_lin = np.r_[np.random.randn(20,2) + [2,2],
               np.random.randn(20,2) + [-2,-2]]
y_lin = np.array([1]*20 + [-1]*20)

# 硬间隔 SVM（C 极大模拟线性可分）
svm_hard = SVC(kernel='linear', C=1e6, random_state=SEED)
svm_hard.fit(X_lin, y_lin)

fig, ax = plt.subplots(figsize=(6, 5.5))

# 绘制样本点
ax.scatter(X_lin[y_lin==1,0],  X_lin[y_lin==1,1],
           color=C_color['cls1'], s=50, edgecolors='white', lw=0.5,
           marker='^', label='类别 +1', zorder=3)
ax.scatter(X_lin[y_lin==-1,0], X_lin[y_lin==-1,1],
           color=C_color['cls0'], s=50, edgecolors='white', lw=0.5,
           label='类别 -1', zorder=3)

# 绘制决策边界和间隔带
w = svm_hard.coef_[0]
b = svm_hard.intercept_[0]
x_plot = np.linspace(X_lin[:,0].min()-0.5, X_lin[:,0].max()+0.5, 100)
ax.plot(x_plot, -(w[0]*x_plot + b)/w[1],
        color='k', lw=2, label='决策边界')
ax.plot(x_plot, -(w[0]*x_plot + b - 1)/w[1],
        color=C_color['neutral'], lw=1.5, ls='--', label='间隔边界')
ax.plot(x_plot, -(w[0]*x_plot + b + 1)/w[1],
        color=C_color['neutral'], lw=1.5, ls='--')

# 填充间隔带
y_up   = -(w[0]*x_plot + b - 1)/w[1]
y_down = -(w[0]*x_plot + b + 1)/w[1]
ax.fill_between(x_plot, y_down, y_up, alpha=0.08, color=C_color['neutral'])

# 标注支持向量
sv = svm_hard.support_vectors_
ax.scatter(sv[:,0],sv[:,1], s=150, facecolors='none',
           edgecolors='k', linewidths=2, zorder=4,
           label=f'支持向量 ({len(sv)}个)')

# 标注间隔宽度
margin = 2 / np.linalg.norm(w)
ax.text(0.05, 0.05, f'间隔宽度 = {margin:.2f}',
        transform=ax.transAxes, fontproperties=FP, fontsize=10,
        color=C_color['neutral'])

ax.set_xlim(X_lin[:,0].min()-0.8, X_lin[:,0].max()+0.8)
ax.set_ylim(X_lin[:,1].min()-0.8, X_lin[:,1].max()+0.8)
ax.set_xlabel('特征 1', fontproperties=FP, fontsize=11)
ax.set_ylabel('特征 2', fontproperties=FP, fontsize=11)
ax.set_title('最大间隔分类器（硬间隔 SVM）', fontproperties=FPB, fontsize=13)
ax.legend(prop=FP, loc='upper left', fontsize=10, framealpha=0.9)
fig.tight_layout()
fig.savefig('figs/fig_D_svm_margin.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_D_svm_margin.png 已保存')

---
## 图2：软间隔 SVM：不同 C 值的决策边界


In [ ]:
# 含噪声的线性不可分数据
X_soft, y_soft = make_classification(
    n_samples=100, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.1, random_state=SEED)

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
fig.subplots_adjust(wspace=0.1)

for ax, C_val in zip(axes, C_values):
    svm_s = SVC(kernel='linear', C=C_val, random_state=SEED)
    svm_s.fit(X_soft, y_soft)
    plot_decision_boundary(ax, svm_s, X_soft, y_soft,
                           title=f'C = {C_val}', show_sv=True)
    n_sv = len(svm_s.support_vectors_)
    ax.text(0.5, 0.02, f'支持向量: {n_sv}',
            transform=ax.transAxes, fontproperties=FP, fontsize=10,
            ha='center', color='k')

axes[0].set_title('C=0.01\n（宽间隔，高偏差）', fontproperties=FPB, fontsize=10)
axes[4].set_title('C=100\n（窄间隔，低偏差）', fontproperties=FPB, fontsize=10)
fig.suptitle('软间隔 SVM：C 从小到大，间隔从宽到窄',
             fontproperties=FPB, fontsize=12, y=1.03)
fig.savefig('figs/fig_D_soft_margin.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_D_soft_margin.png 已保存')

---
## 图3：核方法示意（2D → 3D）


In [ ]:
# 同心圆数据（2D 线性不可分）
X_circ, y_circ = make_circles(n_samples=200, noise=0.1,
                               factor=0.4, random_state=SEED)

fig = plt.figure(figsize=(12, 4.5))

# 左图：2D 原始空间（线性不可分）
ax1 = fig.add_subplot(1,2,1)
ax1.scatter(X_circ[y_circ==0,0], X_circ[y_circ==0,1],
            color=C_color['cls0'], s=30, edgecolors='white', lw=0.4, label='类别 0')
ax1.scatter(X_circ[y_circ==1,0], X_circ[y_circ==1,1],
            color=C_color['cls1'], s=30, edgecolors='white', lw=0.4,
            marker='^', label='类别 1')
ax1.set_title('原始 2D 空间\n（线性不可分）', fontproperties=FPB, fontsize=12)
ax1.set_xlabel('x1', fontproperties=FP); ax1.set_ylabel('x2', fontproperties=FP)
ax1.legend(prop=FP); ax1.set_aspect('equal')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# 右图：3D 变换后的空间（用 phi(x) = [x1, x2, x1^2+x2^2]）
ax2 = fig.add_subplot(1,2,2, projection='3d')
z  = X_circ[:,0]**2 + X_circ[:,1]**2   # 新特征 z = x1^2 + x2^2
ax2.scatter(X_circ[y_circ==0,0], X_circ[y_circ==0,1], z[y_circ==0],
            color=C_color['cls0'], s=25, alpha=0.7, label='类别 0')
ax2.scatter(X_circ[y_circ==1,0], X_circ[y_circ==1,1], z[y_circ==1],
            color=C_color['cls1'], s=25, marker='^', alpha=0.7, label='类别 1')

# 分隔超平面（z = threshold）
threshold = (z[y_circ==0].mean() + z[y_circ==1].mean()) / 2
xx_p = np.linspace(-1.5,1.5,20); yy_p = np.linspace(-1.5,1.5,20)
xx_p, yy_p = np.meshgrid(xx_p, yy_p)
zz_p = np.full_like(xx_p, threshold)
ax2.plot_surface(xx_p, yy_p, zz_p, alpha=0.2, color=C_color['neutral'])

ax2.set_title('映射到 3D 空间\n（线性可分）', fontproperties=FPB, fontsize=12)
ax2.set_xlabel('x1', fontsize=10); ax2.set_ylabel('x2', fontsize=10)
ax2.set_zlabel('x1²+x2²', fontsize=10)
ax2.legend(prop=FP, fontsize=10, loc='upper left')

fig.suptitle('核方法：低维非线性 → 高维线性可分', fontproperties=FPB, fontsize=13)
fig.tight_layout()
fig.savefig('figs/fig_D_kernel_trick.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_D_kernel_trick.png 已保存')

---
## 图4：RBF 核不同 gamma 值下的决策边界


In [ ]:
# moon 形数据（非线性可分）
X_moon, y_moon = make_moons(n_samples=150, noise=0.2, random_state=SEED)
sc_m = StandardScaler()
X_moon_sc = sc_m.fit_transform(X_moon)

gammas = [0.01, 0.1, 1.0, 5.0, 20.0, 100.0]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
fig.subplots_adjust(hspace=0.35, wspace=0.1)

for ax, g in zip(axes.flat, gammas):
    svm_g = SVC(kernel='rbf', C=1.0, gamma=g, random_state=SEED)
    svm_g.fit(X_moon_sc, y_moon)
    train_acc = svm_g.score(X_moon_sc, y_moon)
    plot_decision_boundary(ax, svm_g, X_moon_sc, y_moon,
                           title=f'gamma = {g}', show_sv=False)
    ax.text(0.5, 0.02, f'训练准确率 = {train_acc:.3f}',
            transform=ax.transAxes, fontproperties=FP, fontsize=10,
            ha='center')

fig.suptitle('RBF 核：C=1.0 固定，gamma 从小到大（过小欠拟合，过大过拟合）',
             fontproperties=FPB, fontsize=12)
fig.savefig('figs/fig_D_rbf_gamma.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_D_rbf_gamma.png 已保存')

---
## 图5：SVM vs Logistic 回归决策边界对比


In [ ]:
# 月牙形数据
X_comp, y_comp = make_moons(n_samples=200, noise=0.25, random_state=SEED)
sc_c = StandardScaler()
X_comp_sc = sc_c.fit_transform(X_comp)

models = [
    (LogisticRegression(C=1.0, random_state=SEED), 'Logistic 回归（线性）'),
    (SVC(kernel='linear', C=1.0, random_state=SEED), 'SVM（线性核）'),
    (SVC(kernel='rbf', C=1.0, gamma=0.5, random_state=SEED), 'SVM（RBF 核）'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
fig.subplots_adjust(wspace=0.15)

for ax, (mdl, title) in zip(axes, models):
    mdl.fit(X_comp_sc, y_comp)
    acc = mdl.score(X_comp_sc, y_comp)
    show_sv = isinstance(mdl, SVC)
    plot_decision_boundary(ax, mdl, X_comp_sc, y_comp,
                           title=title, show_sv=show_sv)
    ax.text(0.5, 0.02, f'训练准确率 = {acc:.3f}',
            transform=ax.transAxes, fontproperties=FP, fontsize=10, ha='center')
    if show_sv:
        ax.legend(prop=FP, fontsize=10, loc='upper right')

fig.suptitle('Logistic 回归 vs SVM 决策边界对比（月牙形数据）',
             fontproperties=FPB, fontsize=12)
fig.savefig('figs/fig_D_svm_vs_lr.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_D_svm_vs_lr.png 已保存')

---
## 汇总


In [ ]:
print('\n' + '='*50)
print('Chapter D codes.ipynb 所有图形生成完成')
for f in sorted(os.listdir('figs')):
    if f.startswith('fig_D'):
        print(f'  {f}  ({os.path.getsize("figs/"+f)//1024} KB)')